In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(keyword_fields=["course"], mode="ivf", db_path="faq_vectors2.db")

In [ ]:
query_vector = model.encode("How do I run Docker?")
results = vs_index.search(query_vector, num_results=3, filter_dict={"course": "llm-zoomcamp"})

In [18]:
results

[{'id': 'e8df9f0d12',
  'course': 'llm-zoomcamp',
  'section': 'Module 6: Best Practices',
  'question': 'Docker: When trying to run a streamlit app using docker-compose, I get: Error response from daemon: failed to create task for container: failed to create shim task: OCI runtime create failed: runc create failed: unable to start container process: exec: "streamlit": executable file not found in $PATH: unknown. The app runs fine outside of docker-compose',
  'answer': 'To resolve this issue:\n\n1. Ensure you have created a `Dockerfile`.\n2. Add `streamlit` to the `docker-compose` configuration.\n3. Run the following command to rebuild and start the application:\n\n   ```bash\n   docker-compose up --build\n   ```'},
 {'id': '66ccbb7da0',
  'course': 'llm-zoomcamp',
  'section': 'Module 5: Monitoring',
  'question': 'How can I remove all Docker containers, images, and volumes, and builds from the terminal?',
  'answer': '1. Delete all containers (including running ones):\n\n```bash\ndo

In [12]:
from rag_helper import RAGBase


class RAGVector(RAGBase):
    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict,
        )


In [13]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

openai_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENAI_API_KEY"),
)

In [14]:
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)


In [15]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes, you can still sign up. You can join even after the course has started. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [16]:
vector_assistant.rag("how do i run docker?")

'Use:\n\n```bash\ndocker-compose up --build\n```\n\nIf you’re getting a Streamlit-related Docker error, make sure you have a `Dockerfile` and that `streamlit` is added in the `docker-compose` configuration.'

In [20]:
vs_index.close()